<a href="https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule

Our baseline rule identifies pages that are likely to gain additional organic clicks by improving their search result appearance rather than creating new content.

The rule prioritizes pages that:

Receive a high number of Google Search impressions.
Rank within positions where CTR improvements are realistic (approximately positions 3–15).
Have a lower CTR than expected for their ranking position.

These pages already receive visibility but are not attracting enough clicks, making them good candidates for title and meta description optimization.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("/content/content_refresh_anonymized.csv")
print("DataFrame columns:", df.columns.tolist())

scaler = MinMaxScaler()

df["impressions_norm"] = scaler.fit_transform(df[["impressions_90d"]])

df["position_score"] = (
    15 - df["avg_position"].clip(upper=15)
) / 15

# Expected CTR approximation
df["expected_ctr"] = (
    0.30 - 0.015 * df["avg_position"]
).clip(lower=0.02)

df["actual_ctr"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
).fillna(0)

df["ctr_gap"] = (
    df["expected_ctr"] - df["actual_ctr"]
).clip(lower=0)

df["score"] = (
    0.50 * df["impressions_norm"] +
    0.30 * df["ctr_gap"] +
    0.20 * df["position_score"]
)

df["reason_code"] = np.select(
    [
        (df["impressions_90d"] > 500) & (df["ctr_gap"] > 0.05),
        (df["ctr_gap"] > 0.03),
        (df["impressions_90d"] > 500)
    ],
    [
        "CTR_LOW_HIGH_IMPRESSIONS",
        "CTR_LOW_MEDIUM_POSITION",
        "HIGH_IMPRESSIONS"
    ],
    default="GOOD_CANDIDATE"
)

df["action"] = "Improve CTR"

queue = df.sort_values("score", ascending=False)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

DataFrame columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
CSV written successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20)

for i, row in top20.iterrows():
    print(f"""
Rank: {len(top20[top20.score > row.score])+1}

Action:
{row['action']}

Reason Code:
{row['reason_code']}

Confidence:
High

What would make it wrong?
The page may already have a title/meta update pending, may represent a seasonal query, or impression/CTR data may be too recent to be stable.
""")


Rank: 1

Action:
Improve CTR

Reason Code:
CTR_LOW_HIGH_IMPRESSIONS

Confidence:
High

What would make it wrong?
The page may already have a title/meta update pending, may represent a seasonal query, or impression/CTR data may be too recent to be stable.


Rank: 2

Action:
Improve CTR

Reason Code:
CTR_LOW_HIGH_IMPRESSIONS

Confidence:
High

What would make it wrong?
The page may already have a title/meta update pending, may represent a seasonal query, or impression/CTR data may be too recent to be stable.


Rank: 3

Action:
Improve CTR

Reason Code:
CTR_LOW_HIGH_IMPRESSIONS

Confidence:
High

What would make it wrong?
The page may already have a title/meta update pending, may represent a seasonal query, or impression/CTR data may be too recent to be stable.


Rank: 4

Action:
Improve CTR

Reason Code:
CTR_LOW_HIGH_IMPRESSIONS

Confidence:
High

What would make it wrong?
The page may already have a title/meta update pending, may represent a seasonal query, or impression/CTR data may b

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks

The baseline rule is less reliable for:

Pages with very low impressions because CTR estimates are unstable.
Newly published pages that have insufficient historical search data.
Seasonal or event-driven pages where temporary traffic patterns may distort the score.
Brand-specific queries whose CTR behavior differs from general search queries.
Leakage Check

The baseline rule does not use:

Future click or impression data.
Labels derived from future outcomes.
Existing FlyRank product flags.
Any information unavailable at the decision point.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.